# **⚽ Sistema Experto para Recomendar la Posición de un Jugador**

Sistema experto que analiza las características de un jugador
y recomienda la posición de fútbol que mejor se adapta a su perfil.

In [ ]:
# INSTALACIÓN DE GRADIO
# Instalamos Gradio para poder crear la interfaz
# del sistema experto.
!pip install gradio

In [ ]:
# LIBRERÍAS que vamos a ocupar
# Importamos Gradio para crear la interfaz
# con la que interactuará el usuario.
import gradio as gr

# **Base de Conocimientos**

La base de conocimientos contiene las características y requisitos
asociados con cada posición de fútbol.

La información se obtuvo a partir de la investigación realizada
sobre las diferentes posiciones y posteriormente se organizó
en variables que puede utilizar el sistema experto.

In [ ]:
# BASE DE CONOCIMIENTOS

# La base de conocimientos contiene las características
# que el sistema utiliza para evaluar cada posición.

base_conocimiento = {

    # ------------------------------------------
    # PORTERO
    # ------------------------------------------

    "Portero": {
        "reflejos": {"nivel": "alto", "peso": 2},
        "posicionamiento": {"nivel": "alto", "peso": 2},
        "defensa": {"nivel": "alta", "peso": 2},
        "juego_aereo": {"nivel": "alto", "peso": 2},
        "orientacion": {"nivel": "defensiva", "peso": 2},
        "pase": {"nivel": "medio", "peso": 1},
        "fuerza": {"nivel": "media", "peso": 1},
        "resistencia": {"nivel": "media", "peso": 1}
    },

    # ------------------------------------------
    # DEFENSA CENTRAL
    # ------------------------------------------

    "Defensa central": {
        "defensa": {"nivel": "alta", "peso": 2},
        "fuerza": {"nivel": "alta", "peso": 2},
        "juego_aereo": {"nivel": "alto", "peso": 2},
        "posicionamiento": {"nivel": "alto", "peso": 2},
        "orientacion": {"nivel": "defensiva", "peso": 2},
        "velocidad": {"nivel": "media", "peso": 1},
        "resistencia": {"nivel": "media", "peso": 1},
        "pase": {"nivel": "medio", "peso": 1}
    },

    # ------------------------------------------
    # LATERAL
    # ------------------------------------------

    "Lateral": {
        "velocidad": {"nivel": "alta", "peso": 2},
        "resistencia": {"nivel": "alta", "peso": 2},
        "pase": {"nivel": "alto", "peso": 2},
        "defensa": {"nivel": "alta", "peso": 2},
        "posicionamiento": {"nivel": "alto", "peso": 2},
        "orientacion": {"nivel": "equilibrada", "peso": 2},
        "regate": {"nivel": "medio", "peso": 1},
        "fuerza": {"nivel": "media", "peso": 1}
    },

    # ------------------------------------------
    # MEDIOCAMPISTA DEFENSIVO
    # ------------------------------------------

    "Mediocampista defensivo": {
        "defensa": {"nivel": "alta", "peso": 2},
        "pase": {"nivel": "alto", "peso": 2},
        "resistencia": {"nivel": "alta", "peso": 2},
        "posicionamiento": {"nivel": "alto", "peso": 2},
        "orientacion": {"nivel": "defensiva", "peso": 2},
        "fuerza": {"nivel": "alta", "peso": 1}
    },

    # ------------------------------------------
    # MEDIOCAMPISTA OFENSIVO
    # ------------------------------------------

    "Mediocampista ofensivo": {
        "regate": {"nivel": "alto", "peso": 2},
        "pase": {"nivel": "alto", "peso": 2},
        "posicionamiento": {"nivel": "alto", "peso": 2},
        "orientacion": {"nivel": "ofensiva", "peso": 2},
        "definicion": {"nivel": "media", "peso": 1}
    },

    # ------------------------------------------
    # EXTREMO
    # ------------------------------------------

    "Extremo": {
        "velocidad": {"nivel": "alta", "peso": 2},
        "regate": {"nivel": "alto", "peso": 2},
        "resistencia": {"nivel": "alta", "peso": 2},
        "orientacion": {"nivel": "ofensiva", "peso": 2},
        "definicion": {"nivel": "alta", "peso": 1},
        "pase": {"nivel": "medio", "peso": 1},
        "posicionamiento": {"nivel": "medio", "peso": 1}
    },

    # ------------------------------------------
    # DELANTERO
    # ------------------------------------------

    "Delantero": {
        "definicion": {"nivel": "alta", "peso": 2},
        "posicionamiento": {"nivel": "alto", "peso": 2},
        "fuerza": {"nivel": "alta", "peso": 2},
        "juego_aereo": {"nivel": "alto", "peso": 2},
        "orientacion": {"nivel": "ofensiva", "peso": 2},
        "velocidad": {"nivel": "media", "peso": 1},
        "regate": {"nivel": "medio", "peso": 1}
    }
}

## **Niveles y Reglas**

El sistema utiliza tres niveles para las características cuantificables:

- Bajo = 1
- Medio = 2
- Alto = 3

Estos valores permiten comparar el nivel de una característica
del jugador con el nivel requerido para cada posición.

In [ ]:
# NIVELES Y REGLAS

# Convertimos los niveles cualitativos en valores numéricos
# para poder compararlos dentro del motor de inferencia.

niveles = {
    "baja": 1,
    "media": 2,
    "alta": 3,

    "bajo": 1,
    "medio": 2,
    "alto": 3
}

# **Motor de Inferencia**

El motor de inferencia analiza las características del jugador
y las compara con los requisitos establecidos en la base de conocimientos.

Cada característica que cumple con el requisito de una posición
aporta los puntos correspondientes a su peso.

La posición con mayor puntuación será la recomendación del sistema.

**Evaluación de una posición**

In [ ]:
# Funcion para EVALUAR UNA POSICIÓN

def evaluar_posicion(posicion, jugador):

    # Inicializamos la puntuación en cero.
    puntos = 0

    # Recorremos las características que requiere la posición.
    for caracteristica in posicion:

        # Obtenemos el nivel requerido para esa característica.
        nivel_requerido = posicion[caracteristica]["nivel"]

        # Obtenemos el peso de la característica.
        peso = posicion[caracteristica]["peso"]

        # Obtenemos el nivel que tiene el jugador.
        nivel_jugador = jugador[caracteristica]

        # La orientación se compara directamente porque
        # no utiliza los niveles bajo, medio y alto.
        if caracteristica == "orientacion":

            if nivel_jugador == nivel_requerido:
                puntos += peso

        else:

            # Para las demás características convertimos
            # los niveles a valores numéricos y los comparamos.
            if niveles[nivel_jugador] >= niveles[nivel_requerido]:
                puntos += peso

    # Retorna la puntuación obtenida.
    return puntos

**Evaluar todas las posiciones**

In [ ]:
# Funcion para EVALUAR TODAS LAS POSICIONES

def evaluar_todas_las_posiciones(base_conocimiento, jugador):

    # Diccionario donde guardaremos la puntuación
    # obtenida por cada posición.
    resultados = {}

    # Recorremos todas las posiciones de la base de conocimientos.
    for posicion in base_conocimiento:

        # Evaluamos al jugador respecto a la posición actual.
        puntos = evaluar_posicion(base_conocimiento[posicion],jugador)

        # Guardamos la puntuación.
        resultados[posicion] = puntos

    return resultados

**Características principales**

Además de la puntuación total, el sistema cuenta cuántas
características principales de cada posición cumple el jugador.

Las características con peso 2 se consideran principales.

Esta información también se utiliza para resolver empates
entre posiciones con la misma puntuación.

In [ ]:
# CONTAR CARACTERÍSTICAS PRINCIPALES

def contar_caracteristicas_principales(posicion, jugador):

    # Contador de características principales que cumple el jugador.
    principales = 0

    # Recorremos las características de la posición.
    for caracteristica in posicion:

        # Obtenemos el peso de la característica.
        peso = posicion[caracteristica]["peso"]

        # Solo contamos las características con peso 2,
        # ya que representan las características principales.
        if peso == 2:

            nivel_requerido = posicion[caracteristica]["nivel"]
            nivel_jugador = jugador[caracteristica]

            # La orientación se compara directamente.
            if caracteristica == "orientacion":

                if nivel_jugador == nivel_requerido:
                    principales += 1

            else:

                # Las demás características se comparan
                # mediante sus valores numéricos.
                if niveles[nivel_jugador] >= niveles[nivel_requerido]:
                    principales += 1

    return principales

**Recomendar posición**

Esta función reúne las puntuaciones obtenidas para las siete posiciones
y determina cuál tiene la mayor coincidencia con el perfil del jugador.

Si existe un empate, se utiliza como segundo criterio la cantidad
de características principales que cumple el jugador.

In [ ]:
# Funcion para RECOMENDAR POSICIÓN

def recomendar_posicion(base_conocimiento, jugador):

    # Evaluamos al jugador en todas las posiciones.
    resultados = evaluar_todas_las_posiciones(base_conocimiento,jugador)

    # Guardaremos la cantidad de características principales
    # que cumple el jugador para cada posición.
    principales = {}

    for posicion in base_conocimiento:

        cantidad = contar_caracteristicas_principales(base_conocimiento[posicion],jugador)

        principales[posicion] = cantidad

    # Obtenemos la puntuación más alta.
    mayor_puntaje = max(resultados.values())

    # Buscamos todas las posiciones que obtuvieron
    # esa puntuación.
    candidatas = []

    for posicion in resultados:

        if resultados[posicion] == mayor_puntaje:
            candidatas.append(posicion)

    # Si solamente hay una posición con la puntuación más alta,
    # esa es nuestra recomendación.
    if len(candidatas) == 1:

        return candidatas[0], resultados, principales

    # Si existe un empate, tomamos inicialmente
    # la primera posición candidata.
    mejor_posicion = candidatas[0]

    # Comparamos las demás posiciones empatadas.
    for posicion in candidatas[1:]:

        # Si una posición cumple más características principales,
        # pasa a ser la mejor candidata.
        if principales[posicion] > principales[mejor_posicion]:

            mejor_posicion = posicion

    return mejor_posicion, resultados, principales

**Funciones auxiliares**

Estas funciones complementan al motor de inferencia.

Se utilizan para identificar las características que coinciden
con la posición recomendada y para convertir los nombres técnicos
de las variables a nombres más fáciles de mostrar al usuario.

In [ ]:
# Funcion para OBTENER CARACTERÍSTICAS QUE COINCIDEN
# Sirve para mostrar despues en la interfaz

def obtener_caracteristicas_coincidentes(posicion, jugador):

    # Lista donde guardaremos las características
    # que cumplen con el requisito de la posición.
    coincidencias = []

    # Recorremos las características de la posición recomendada.
    for caracteristica in posicion:

        nivel_requerido = posicion[caracteristica]["nivel"]
        nivel_jugador = jugador[caracteristica]

        # La orientación se compara directamente.
        if caracteristica == "orientacion":

            if nivel_jugador == nivel_requerido:
                coincidencias.append(caracteristica)

        else:

            # Las demás características se comparan
            # mediante sus valores numéricos.
            if niveles[nivel_jugador] >= niveles[nivel_requerido]:
                coincidencias.append(caracteristica)

    return coincidencias

In [ ]:
# NOMBRES AMIGABLES

# Convertimos los nombres utilizados internamente
# por Python a nombres más fáciles de leer para el usuario.

nombres_caracteristicas = {
    "velocidad": "Velocidad",
    "resistencia": "Resistencia",
    "regate": "Regate",
    "pase": "Pase",
    "defensa": "Defensa",
    "definicion": "Definición",
    "fuerza": "Fuerza",
    "juego_aereo": "Juego aéreo",
    "reflejos": "Reflejos",
    "posicionamiento": "Posicionamiento",
    "orientacion": "Orientación"
}

**Sistema experto**

El sistema experto recibe el perfil de un jugador y utiliza
la base de conocimientos y el motor de inferencia para obtener
una posición recomendada.

También obtiene las características que coincidieron con
la posición recomendada.

In [ ]:
# SISTEMA EXPERTO

def sistema_experto(jugador):

    # Ejecutamos el motor de inferencia para obtener
    # la posición recomendada y las puntuaciones.
    posicion, resultados, principales = recomendar_posicion(
        base_conocimiento,
        jugador
    )

    # Obtenemos las características que coincidieron
    # con la posición recomendada.
    coincidencias = obtener_caracteristicas_coincidentes(
        base_conocimiento[posicion],
        jugador
    )

    return posicion, resultados, principales, coincidencias

**Pruebas del sistema**

Se realizaron pruebas con perfiles representativos de las siete
posiciones consideradas por el sistema experto.

El objetivo es comprobar que el motor de inferencia identifica
correctamente la posición correspondiente a cada perfil.

In [ ]:
# PRUEBA 1: PORTERO

jugador_portero = {
    "velocidad": "baja",
    "resistencia": "media",
    "regate": "bajo",
    "pase": "medio",
    "defensa": "alta",
    "definicion": "baja",
    "fuerza": "media",
    "juego_aereo": "alto",
    "reflejos": "alto",
    "posicionamiento": "alto",
    "orientacion": "defensiva"
}

sistema_experto(jugador_portero)

('Portero',
 {'Portero': 13,
  'Defensa central': 10,
  'Lateral': 5,
  'Mediocampista defensivo': 6,
  'Mediocampista ofensivo': 2,
  'Extremo': 2,
  'Delantero': 4},
 {'Portero': 5,
  'Defensa central': 4,
  'Lateral': 2,
  'Mediocampista defensivo': 3,
  'Mediocampista ofensivo': 1,
  'Extremo': 0,
  'Delantero': 2},
 ['reflejos',
  'posicionamiento',
  'defensa',
  'juego_aereo',
  'orientacion',
  'pase',
  'fuerza',
  'resistencia'])

In [ ]:
# PRUEBA 2: DEFENSA CENTRAL

jugador_defensa = {
    "velocidad": "media",
    "resistencia": "media",
    "regate": "bajo",
    "pase": "medio",
    "defensa": "alta",
    "definicion": "baja",
    "fuerza": "alta",
    "juego_aereo": "alto",
    "reflejos": "bajo",
    "posicionamiento": "alto",
    "orientacion": "defensiva"
}

sistema_experto(jugador_defensa)

('Defensa central',
 {'Portero': 11,
  'Defensa central': 13,
  'Lateral': 5,
  'Mediocampista defensivo': 7,
  'Mediocampista ofensivo': 2,
  'Extremo': 2,
  'Delantero': 7},
 {'Portero': 4,
  'Defensa central': 5,
  'Lateral': 2,
  'Mediocampista defensivo': 3,
  'Mediocampista ofensivo': 1,
  'Extremo': 0,
  'Delantero': 3},
 ['defensa',
  'fuerza',
  'juego_aereo',
  'posicionamiento',
  'orientacion',
  'velocidad',
  'resistencia',
  'pase'])

In [ ]:
# PRUEBA 3: LATERAL

jugador_lateral = {
    "velocidad": "alta",
    "resistencia": "alta",
    "regate": "medio",
    "pase": "alto",
    "defensa": "alta",
    "definicion": "baja",
    "fuerza": "media",
    "juego_aereo": "medio",
    "reflejos": "bajo",
    "posicionamiento": "alto",
    "orientacion": "equilibrada"
}

sistema_experto(jugador_lateral)

('Lateral',
 {'Portero': 7,
  'Defensa central': 7,
  'Lateral': 14,
  'Mediocampista defensivo': 8,
  'Mediocampista ofensivo': 4,
  'Extremo': 6,
  'Delantero': 4},
 {'Portero': 2,
  'Defensa central': 2,
  'Lateral': 6,
  'Mediocampista defensivo': 4,
  'Mediocampista ofensivo': 2,
  'Extremo': 2,
  'Delantero': 1},
 ['velocidad',
  'resistencia',
  'pase',
  'defensa',
  'posicionamiento',
  'orientacion',
  'regate',
  'fuerza'])

In [ ]:
# PRUEBA 4: MEDIOCAMPISTA DEFENSIVO

jugador_mcd = {
    "velocidad": "media",
    "resistencia": "alta",
    "regate": "bajo",
    "pase": "alto",
    "defensa": "alta",
    "definicion": "baja",
    "fuerza": "alta",
    "juego_aereo": "medio",
    "reflejos": "bajo",
    "posicionamiento": "alto",
    "orientacion": "defensiva"
}

sistema_experto(jugador_mcd)

('Mediocampista defensivo',
 {'Portero': 9,
  'Defensa central': 11,
  'Lateral': 9,
  'Mediocampista defensivo': 11,
  'Mediocampista ofensivo': 4,
  'Extremo': 4,
  'Delantero': 5},
 {'Portero': 3,
  'Defensa central': 4,
  'Lateral': 4,
  'Mediocampista defensivo': 5,
  'Mediocampista ofensivo': 2,
  'Extremo': 1,
  'Delantero': 2},
 ['defensa',
  'pase',
  'resistencia',
  'posicionamiento',
  'orientacion',
  'fuerza'])

In [ ]:
# PRUEBA 5: MEDIOCAMPISTA OFENSIVO

jugador_mco = {
    "velocidad": "media",
    "resistencia": "media",
    "regate": "alto",
    "pase": "alto",
    "defensa": "baja",
    "definicion": "media",
    "fuerza": "baja",
    "juego_aereo": "bajo",
    "reflejos": "bajo",
    "posicionamiento": "alto",
    "orientacion": "ofensiva"
}

sistema_experto(jugador_mco)

('Mediocampista ofensivo',
 {'Portero': 4,
  'Defensa central': 5,
  'Lateral': 5,
  'Mediocampista defensivo': 4,
  'Mediocampista ofensivo': 9,
  'Extremo': 6,
  'Delantero': 6},
 {'Portero': 1,
  'Defensa central': 1,
  'Lateral': 2,
  'Mediocampista defensivo': 2,
  'Mediocampista ofensivo': 4,
  'Extremo': 2,
  'Delantero': 2},
 ['regate', 'pase', 'posicionamiento', 'orientacion', 'definicion'])

In [ ]:
# PRUEBA 6: EXTREMO

jugador_extremo = {
    "velocidad": "alta",
    "resistencia": "alta",
    "regate": "alto",
    "pase": "medio",
    "defensa": "baja",
    "definicion": "alta",
    "fuerza": "baja",
    "juego_aereo": "bajo",
    "reflejos": "bajo",
    "posicionamiento": "medio",
    "orientacion": "ofensiva"
}

sistema_experto(jugador_extremo)

('Extremo',
 {'Portero': 2,
  'Defensa central': 3,
  'Lateral': 5,
  'Mediocampista defensivo': 2,
  'Mediocampista ofensivo': 5,
  'Extremo': 11,
  'Delantero': 6},
 {'Portero': 0,
  'Defensa central': 0,
  'Lateral': 2,
  'Mediocampista defensivo': 1,
  'Mediocampista ofensivo': 2,
  'Extremo': 4,
  'Delantero': 2},
 ['velocidad',
  'regate',
  'resistencia',
  'orientacion',
  'definicion',
  'pase',
  'posicionamiento'])

In [ ]:
# PRUEBA 7: DELANTERO

jugador_delantero = {
    "velocidad": "media",
    "resistencia": "media",
    "regate": "medio",
    "pase": "bajo",
    "defensa": "baja",
    "definicion": "alta",
    "fuerza": "alta",
    "juego_aereo": "alto",
    "reflejos": "bajo",
    "posicionamiento": "alto",
    "orientacion": "ofensiva"
}

sistema_experto(jugador_delantero)

('Delantero',
 {'Portero': 6,
  'Defensa central': 8,
  'Lateral': 4,
  'Mediocampista defensivo': 3,
  'Mediocampista ofensivo': 5,
  'Extremo': 4,
  'Delantero': 12},
 {'Portero': 2,
  'Defensa central': 3,
  'Lateral': 1,
  'Mediocampista defensivo': 1,
  'Mediocampista ofensivo': 2,
  'Extremo': 1,
  'Delantero': 5},
 ['definicion',
  'posicionamiento',
  'fuerza',
  'juego_aereo',
  'orientacion',
  'velocidad',
  'regate'])

**Interfaz de usuario**

La interfaz permite que el usuario ingrese las características
del jugador mediante menús desplegables.

Las respuestas se convierten en un perfil y se envían al sistema
experto para obtener la posición recomendada.

Para construir la interfaz utilizamos Gradio.

In [ ]:
# FUNCIÓN PARA CONECTAR LA INTERFAZ

def ejecutar_sistema(
    velocidad,
    resistencia,
    regate,
    pase,
    defensa,
    definicion,
    fuerza,
    juego_aereo,
    reflejos,
    posicionamiento,
    orientacion
):

    # Verificamos que el usuario haya respondido
    # todas las características.
    respuestas = [
        velocidad,
        resistencia,
        regate,
        pase,
        defensa,
        definicion,
        fuerza,
        juego_aereo,
        reflejos,
        posicionamiento,
        orientacion
    ]

    if None in respuestas:

        return "Completa todas las características del jugador."

    # Creamos el perfil del jugador utilizando
    # las respuestas proporcionadas en la interfaz.
    jugador = {
        "velocidad": velocidad,
        "resistencia": resistencia,
        "regate": regate,
        "pase": pase,
        "defensa": defensa,
        "definicion": definicion,
        "fuerza": fuerza,
        "juego_aereo": juego_aereo,
        "reflejos": reflejos,
        "posicionamiento": posicionamiento,
        "orientacion": orientacion
    }

    # Ejecutamos el sistema experto con el perfil del jugador.
    posicion, resultados, principales, coincidencias = sistema_experto(
        jugador
    )

    # Convertimos los nombres técnicos de Python
    # a nombres más fáciles de leer.
    nombres = [
        nombres_caracteristicas[c]
        for c in coincidencias
    ]

    # Creamos el texto que verá el usuario.
    resultado = f"""
POSICIÓN RECOMENDADA

{posicion.upper()}

Puntuación: {resultados[posicion]} puntos

Características que coinciden:
"""

    # Agregamos las características coincidentes
    # al resultado final.
    for nombre in nombres:
        resultado += f"\n- {nombre}"

    return resultado

**Diseño de la interfaz**

In [ ]:
# DISEÑO DE LA INTERFAZ

with gr.Blocks(
    title="Sistema Experto - Posición de Fútbol"
) as interfaz:

    # ==========================================
    # ENCABEZADO
    # ==========================================

    gr.Markdown(
        """
        <div class="titulo">

        # ⚽ Sistema Experto

        ## Recomendación de posición de un jugador

        </div>

        <div class="subtitulo">

        Ingresa las características del jugador y el sistema analizará
        cuál es la posición que mejor se adapta a su perfil.

        </div>
        """
    )

    # ==========================================
    # CARACTERÍSTICAS FÍSICAS
    # ==========================================

    gr.Markdown("### Características físicas")

    with gr.Row(elem_classes="seccion"):

        velocidad = gr.Dropdown(
            choices=["baja", "media", "alta"],
            value=None,
            label="Velocidad"
        )

        resistencia = gr.Dropdown(
            choices=["baja", "media", "alta"],
            value=None,
            label="Resistencia"
        )

        fuerza = gr.Dropdown(
            choices=["baja", "media", "alta"],
            value=None,
            label="Fuerza"
        )

    # ==========================================
    # CARACTERÍSTICAS TÉCNICAS
    # ==========================================

    gr.Markdown("### Características técnicas")

    with gr.Row(elem_classes="seccion"):

        regate = gr.Dropdown(
            choices=["bajo", "medio", "alto"],
            value=None,
            label="Regate"
        )

        pase = gr.Dropdown(
            choices=["bajo", "medio", "alto"],
            value=None,
            label="Pase"
        )

        definicion = gr.Dropdown(
            choices=["baja", "media", "alta"],
            value=None,
            label="Definición"
        )

        juego_aereo = gr.Dropdown(
            choices=["bajo", "medio", "alto"],
            value=None,
            label="Juego aéreo"
        )

    # ==========================================
    # CARACTERÍSTICAS TÁCTICAS
    # ==========================================

    gr.Markdown("### Características tácticas")

    with gr.Row(elem_classes="seccion"):

        defensa = gr.Dropdown(
            choices=["baja", "media", "alta"],
            value=None,
            label="Defensa"
        )

        posicionamiento = gr.Dropdown(
            choices=["bajo", "medio", "alto"],
            value=None,
            label="Posicionamiento"
        )

        reflejos = gr.Dropdown(
            choices=["bajo", "medio", "alto"],
            value=None,
            label="Reflejos"
        )

        orientacion = gr.Dropdown(
            choices=["defensiva", "equilibrada", "ofensiva"],
            value=None,
            label="Orientación"
        )

    # ==========================================
    # BOTÓN
    # ==========================================

    boton = gr.Button(
        "RECOMENDAR POSICIÓN",
        variant="primary",
        elem_classes="boton"
    )

    # ==========================================
    # RESULTADO
    # ==========================================

    gr.Markdown("### Resultado")

    resultado = gr.Textbox(
        label="Análisis del sistema experto",
        interactive=False,
        lines=12,
        elem_classes="resultado"
    )

    # ==========================================
    # CONEXIÓN CON EL SISTEMA EXPERTO
    # ==========================================

    boton.click(
        fn=ejecutar_sistema,
        inputs=[
            velocidad,
            resistencia,
            regate,
            pase,
            defensa,
            definicion,
            fuerza,
            juego_aereo,
            reflejos,
            posicionamiento,
            orientacion
        ],
        outputs=resultado
    )

**Ejecutar la interfaz**

In [ ]:
# EJECUTAR LA INTERFAZ

interfaz.launch(
    theme=gr.themes.Soft(
        primary_hue="green",
        secondary_hue="gray",
        neutral_hue="slate"
    ),
    css="""
    body {
        background-color: #0f172a;
    }

    .titulo {
        text-align: center;
        margin-bottom: 10px;
    }

    .subtitulo {
        text-align: center;
        color: #94a3b8;
        margin-bottom: 25px;
    }

    .seccion {
        border-radius: 12px;
        padding: 10px;
    }

    .resultado {
        border: 2px solid #22c55e;
        border-radius: 12px;
    }

    .boton {
        margin-top: 15px;
        margin-bottom: 15px;
    }
    """
)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb9471d1f506c4768c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
